In [27]:
import numpy as np
from scipy import spatial as sp
from itertools import product, permutations
from matplotlib import pyplot as plt

from point_cloud import PointCloud


Let $A, B \in \mathbb{R}^k$ be the two point clouds for $k \in \{2, 3\}$ centered at the origin, and let $d_\max = \max\{\text{diam} A, \text{diam} B\}$. Because the distance from the origin to the most extreme point of $A\cup B$ is $< d_\max$, which is invariant to rotation or reflection, we never need to translate $A$ by more than $d_\max$. Therefore, its points will always be contained in the bounding box
$[-2d_{\max}, 2d_{\max}]^{k} \subset \mathbb{R}^k$. We use this box to obtain bounded Voronoi cells for quick point location when computing $d_\text{H}(T(A), B)$.

In [28]:
def diam(coords):
    hull = sp.ConvexHull(coords)
    hull_coords = coords[hull.vertices]
    candidate_distances = sp.distance.cdist(hull_coords, hull_coords)
    
    return candidate_distances.max()


def hausDist(A,B):
    m = sp.distance.directed_hausdorff(A, B)[0]
    n = sp.distance.directed_hausdorff(B, A)[0]
    return max(m,n)

In [29]:
%%time
np.random.seed(10)
n_dim = 2
n = 100000
n_A, n_B = n, n

# Generate points.
coords_A_and_B = [np.random.rand(n, n_dim) for n in [n_A, n_B]]

# Compute the bounding box.
max_diam = max(map(diam, coords_A_and_B))
bbox = [np.array([-2*max_diam, 2*max_diam]) for _ in range(n_dim)]

# Construct the two point clouds.
A, B = (PointCloud(coords, bbox) for coords in coords_A_and_B)

CPU times: user 1min 2s, sys: 738 ms, total: 1min 3s
Wall time: 1min 2s


Hausdorff distance between A and B:


In [32]:
%%time
A.dH(B)

CPU times: user 1.22 s, sys: 514 ms, total: 1.74 s
Wall time: 1.74 s


0.007104935196016974

In [33]:
%%time
hausDist(A.points.coords, B.points.coords)

CPU times: user 5.06 s, sys: 15.6 ms, total: 5.07 s
Wall time: 5.06 s


0.007104935196016974

Transforming A by reflecting and rotating by 90°:

In [34]:
%%time
TA = A.transform(nontriv_refl=True, theta=np.pi/2)

CPU times: user 30.3 s, sys: 260 ms, total: 30.5 s
Wall time: 30.3 s


Hausdorff distance between transformed A and B:

In [37]:
%%time
TA.dH(B)

CPU times: user 1.19 s, sys: 514 ms, total: 1.7 s
Wall time: 1.71 s


0.007787187857478472

In [38]:
%%time
hausDist(TA.points.coords, B.points.coords)

CPU times: user 4.15 s, sys: 11.4 ms, total: 4.16 s
Wall time: 4.15 s


0.007787187857478472